In [ ]:
import pandas as pd
import numpy as np
df = pd.read_csv(r"D:\DS115-VIS\.venv\Project1_Predictive-Maintenance\ai4i_fused.csv")
df.head()


In [ ]:
# Clean trailing and leading spaces from column names
df.columns = df.columns.str.strip()

In [ ]:
# 2. Sort data for time-series compatibility
# Chronological order is mandatory for accurate rolling calculations
df = df.sort_values(by=['date', 'UDI']).reset_index(drop=True)

In [ ]:
# 3. Select Target Telemetry Features
# These are the main continuous sensor and weather columns from image
sensor_features = [
    'Air tempe', 'Process te', 'Rotationa', 'Torque', 'Tool wear',
    'avg_temp', 'min_temp', 'max_temp', 'precipitat', 'avg_wind', 'avg_sea_level_pres_hpa'
]

In [ ]:
# Define the rolling window size (Operational window size - e.g., 5 logs)
window_size = 5
df.head(10)

In [ ]:
print("Starting feature engineering...")


In [ ]:
# 4. Engineer Rolling Mean, Standard Deviation, and Variance
# Perform calculations inside a new DataFrame
engineered_features = pd.DataFrame(index=df.index)
df


In [ ]:
for col in sensor_features:
    # Check if the column exists in the dataset
    if col in df.columns:
        # Rolling Mean
        engineered_features[f'{col}_roll_mean'] = df[col].rolling(window=window_size).mean()
        
        # Rolling Standard Deviation
        engineered_features[f'{col}_roll_std'] = df[col].rolling(window=window_size).std()
        
        # Rolling Variance
        engineered_features[f'{col}_roll_var'] = df[col].rolling(window=window_size).var()
df

In [ ]:
# 5. Combine with original metadata and targets
# Includes UDI, Product ID, Machine failure, and specific failure modes
meta_cols = ['UDI', 'Product ID', 'Type', 'date', 'Machine f', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']
# Filter out names to match what is present in the dataset
meta_cols = [c for c in meta_cols if c in df.columns]

final_df = pd.concat([df[meta_cols], engineered_features], axis=1)


In [ ]:
# 6. Drop NaN rows generated by the rolling window warmup buffer
final_df.dropna(inplace=True)
df.head(10)


In [ ]:
# 7. Verify the output structure and save the file
print(f"Processed dataset shape: {final_df.shape}")
print(final_df.head())

# Save the final engineered dataset
final_df.to_csv("engineered_iot_weather_dataset.csv", index=False)
print("File successfully saved as 'engineered_iot_weather_dataset.csv'!")
